In [1]:
!pip install joblib

In [1]:
import joblib
from datetime import datetime

# Save model and metadata
model_version = "1.0.0"
model_dir = Path("../models") / model_version
model_dir.mkdir(parents=True, exist_ok=True)

# Save pipeline
model_path = model_dir / "pipeline.joblib"
joblib.dump(best_model, model_path)
print(f"✓ Model saved to {model_path}")

# Save metadata
metadata = {
    "model_id": f"resume_screening_v{model_version}",
    "model_type": "Logistic Regression",
    "version": model_version,
    "created_at": datetime.utcnow().isoformat(),
    "training_date": datetime.utcnow().isoformat(),
    "main_metric": {
        "f1_score": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "roc_auc": float(roc_auc),
    },
    "training_samples": len(X_train),
    "test_samples": len(X_test),
    "feature_count": len(feature_names),
    "hyperparameters": {
        "C": float(grid_search.best_params_['classifier__C']),
        "solver": grid_search.best_params_['classifier__solver'],
        "max_iter": int(grid_search.best_params_['classifier__max_iter']),
        "tfidf_max_features": 5000,
        "tfidf_ngram_range": (1, 2),
        "tfidf_min_df": 2,
        "tfidf_max_df": 0.95,
    },
    "preprocessing": {
        "lowercase": True,
        "remove_stopwords": True,
        "lemmatize": True,
        "min_token_length": 2,
    },
    "notes": "Production-ready Logistic Regression model for resume-job matching. Optimized for CPU inference.",
}

metadata_path = model_dir / "metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved to {metadata_path}")

print("\n" + "="*80)
print("🎉 MODEL TRAINING COMPLETE!")
print("="*80)
print(f"Model version: {model_version}")
print(f"Location: {model_dir}")
print(f"\nPerformance:")
print(f"  F1-Score: {f1:.3f}")
print(f"  Precision: {precision:.3f}")
print(f"  Recall: {recall:.3f}")
print(f"  ROC-AUC: {roc_auc:.3f}")
print(f"\nReady for deployment! 🚀")

NameError: name 'Path' is not defined

## Model Persistence & Versioning

Save the trained model for production deployment, including metadata for reproducibility.

In [ ]:
# Extract feature importance (interpretability)
tfidf_vectorizer = best_model.named_steps['tfidf']
classifier = best_model.named_steps['classifier']

feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
coefficients = classifier.coef_[0]

# Get top positive and negative features
top_k = 15
top_positive_idx = np.argsort(coefficients)[-top_k:][::-1]
top_negative_idx = np.argsort(coefficients)[:top_k]

print("\n🔍 FEATURE IMPORTANCE (Model Explainability)")
print("="*80)

print(f"\nTop {top_k} Features INDICATING MATCH (positive coefficient):")
for i, idx in enumerate(top_positive_idx, 1):
    print(f"  {i:2d}. {feature_names[idx]:30s} coef={coefficients[idx]:7.4f}")

print(f"\nTop {top_k} Features INDICATING NON-MATCH (negative coefficient):")
for i, idx in enumerate(top_negative_idx, 1):
    print(f"  {i:2d}. {feature_names[idx]:30s} coef={coefficients[idx]:7.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top positive
top_positive_features = feature_names[top_positive_idx]
top_positive_coefs = coefficients[top_positive_idx]
axes[0].barh(top_positive_features, top_positive_coefs, color='green', alpha=0.7)
axes[0].set_title(f'Top {top_k} Match Indicators')
axes[0].set_xlabel('Coefficient Value')

# Top negative
top_negative_features = feature_names[top_negative_idx]
top_negative_coefs = coefficients[top_negative_idx]
axes[1].barh(top_negative_features, -top_negative_coefs, color='red', alpha=0.7)
axes[1].set_title(f'Top {top_k} Non-Match Indicators')
axes[1].set_xlabel('Coefficient Value (absolute)')

plt.tight_layout()
plt.show()

## Model Explainability: Top Important Features

Extract the most important words (features) that the model learned are signals for matches.
Positive coefficient = indicates a match, Negative coefficient = indicates a non-match.

In [ ]:
# Final evaluation on test set
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate metrics
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
cm = confusion_matrix(y_test, y_pred)

print("📊 FINAL MODEL EVALUATION")
print("="*80)
print(f"\nPrecision: {precision:.3f} (of predictions, {precision:.1%} are correct)")
print(f"Recall:    {recall:.3f} (we find {recall:.1%} of true matches)")
print(f"F1-Score:  {f1:.3f} (balanced score)")
print(f"ROC-AUC:   {roc_auc:.3f} (0.5 = random, 1.0 = perfect)")

print("\nConfusion Matrix:")
print(f"  True Negatives:  {cm[0, 0]:3d} (correctly rejected)")
print(f"  False Positives: {cm[0, 1]:3d} (wrongly accepted)")
print(f"  False Negatives: {cm[1, 0]:3d} (wrongly rejected) ⚠️ BAD IN HIRING")
print(f"  True Positives:  {cm[1, 1]:3d} (correctly accepted)")

# Detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Match', 'Match']))

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Evaluation', fontsize=16, fontweight='bold')

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], cbar=False)
axes[0, 0].set_title('Confusion Matrix')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')
axes[0, 0].set_xticklabels(['No Match', 'Match'])
axes[0, 0].set_yticklabels(['No Match', 'Match'])

# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
axes[0, 1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc:.3f}')
axes[0, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random (AUC=0.5)')
axes[0, 1].set_xlim([0.0, 1.0])
axes[0, 1].set_ylim([0.0, 1.05])
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve')
axes[0, 1].legend(loc="lower right")

# 3. Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1, 0].plot(recall_curve, precision_curve, color='green', lw=2)
axes[1, 0].fill_between(recall_curve, precision_curve, alpha=0.2, color='green')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curve')
axes[1, 0].set_xlim([0.0, 1.0])
axes[1, 0].set_ylim([0.0, 1.05])
axes[1, 0].grid(alpha=0.3)

# 4. Metrics Summary
metrics_summary = [
    ('Precision', precision),
    ('Recall', recall),
    ('F1-Score', f1),
    ('ROC-AUC', roc_auc),
]
metric_names = [m[0] for m in metrics_summary]
metric_values = [m[1] for m in metrics_summary]
colors_metrics = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

axes[1, 1].barh(metric_names, metric_values, color=colors_metrics, edgecolor='black')
axes[1, 1].set_xlim([0, 1])
axes[1, 1].set_title('Metrics Summary')
axes[1, 1].set_xlabel('Score')
for i, v in enumerate(metric_values):
    axes[1, 1].text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

## Final Model Evaluation

We evaluate the best model on the test set using multiple metrics:
- **Precision**: Of predicted matches, how many are actually matches?
- **Recall**: Of actual matches, how many did we find?
- **F1-Score**: Harmonic mean of precision & recall
- **ROC-AUC**: Overall classification performance

**For hiring context**: 
- High Precision = fewer false positives (don't waste HR time with wrong matches)
- High Recall = fewer false negatives (don't miss good candidates)
- We prefer balanced F1-score for both

In [ ]:
# Hyperparameter tuning for Logistic Regression
print("\n🎯 HYPERPARAMETER TUNING (Logistic Regression)")
print("="*80)

# Define pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced')),
])

# Define parameter grid
param_grid = {
    'classifier__C': [0.001, 0.01, 0.1, 1, 10],
    'classifier__solver': ['lbfgs', 'liblinear'],
    'classifier__max_iter': [1000, 2000],
}

# GridSearchCV with stratified K-fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=skf,
    scoring='f1',  # Optimize for F1-score
    n_jobs=-1,  # Use all CPU cores
    verbose=1,
)

print("Training GridSearchCV (this may take ~30-60s)...")
start = time.time()
grid_search.fit(X_train, y_train)
tuning_time = time.time() - start

print(f"\n✓ Tuning complete in {tuning_time:.2f}s")
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1-score: {grid_search.best_score_:.3f}")

# Results DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)
results_df_sorted = results_df.sort_values('rank_test_score')[['param_classifier__C', 'param_classifier__solver', 
                                                                   'param_classifier__max_iter', 'mean_test_score', 'rank_test_score']]
print("\nTop 5 parameter combinations:")
print(results_df_sorted.head(5).to_string())

## Hyperparameter Tuning with GridSearchCV

We'll tune the best model (Logistic Regression) using stratified cross-validation.

Key parameters:
- **C**: Inverse regularization strength (smaller = stronger regularization)
- **solver**: Optimization algorithm (lbfgs, liblinear, saga)
- **max_iter**: Max iterations to reach convergence

In [ ]:
# Quick model comparison (using default parameters)
print("🔍 MODEL COMPARISON")
print("="*80)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    'Linear SVM': LinearSVC(random_state=42, dual=False, class_weight='balanced', max_iter=2000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'),
}

comparison_results = {}

for model_name, model in models.items():
    # Create pipeline
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ('classifier', model),
    ])
    
    # Time the training
    start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Predict
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline.named_steps['classifier'], 'predict_proba') else y_pred
    
    # Evaluate
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    comparison_results[model_name] = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'train_time': train_time,
        'pipeline': pipeline,
    }
    
    print(f"\n{model_name}:")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall: {recall:.3f}")
    print(f"  F1-Score: {f1:.3f}")
    print(f"  ROC-AUC: {roc_auc:.3f}")
    print(f"  Training time: {train_time:.2f}s")

# Display comparison table
comparison_df = pd.DataFrame({
    model: {metric: value for metric, value in results.items() if metric != 'pipeline'}
    for model, results in comparison_results.items()
}).T

print("\n" + "="*80)
print("COMPARISON TABLE:")
print(comparison_df.to_string())

# Recommend best model
best_model_name = max(comparison_results.items(), key=lambda x: x[1]['f1'])[0]
print(f"\n✓ RECOMMENDATION: {best_model_name}")
print(f"  Reason: Best F1-score ({comparison_results[best_model_name]['f1']:.3f})")
print(f"  Fast training ({comparison_results[best_model_name]['train_time']:.2f}s)")
print(f"  Interpretable coefficients for explainability")

## Model Comparison

We'll compare three models on sparse TF-IDF data:

| Model | Pros | Cons | Best For |
|-------|------|------|----------|
| **Logistic Regression** | Fast, interpretable, handles sparse data well | Linear assumption | ✓ **RECOMMENDED** |
| **Linear SVM** | Large margin, handles high dimensions | Slower training, less interpretable | Scalability |
| **Random Forest** | Non-linear, feature importance | Slow on sparse data, high memory | Baseline (not recommended) |

**Our choice: Logistic Regression** because:
- ✓ Fastest training & inference on CPU
- ✓ Coefficients directly show which words matter (explainability)
- ✓ Works great with sparse TF-IDF vectors
- ✓ Easy to deploy and maintain

In [ ]:
# Train/Test Split (Stratified to maintain class balance)
X_train, X_test, y_train, y_test = train_test_split(
    processed_texts,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels,  # Ensures same class ratio in train/test
)

print(f"Training set: {len(X_train)} samples (Positive: {sum(y_train)}, Negative: {len(y_train) - sum(y_train)})")
print(f"Test set: {len(X_test)} samples (Positive: {sum(y_test)}, Negative: {len(y_test) - sum(y_test)})")

In [ ]:
# Load data (previously generated in EDA notebook)
with open("../data/resumes.json") as f:
    resumes = json.load(f)

with open("../data/jobs.json") as f:
    jobs = json.load(f)

with open("../data/labels.json") as f:
    labels = json.load(f)

print(f"Loaded {len(resumes)} resumes, {len(jobs)} jobs, {len(labels)} labels")

# Convert to DataFrame for easier manipulation
labels_df = pd.DataFrame(labels)

# Get texts for each resume-job pair
texts = []
y_labels = []

for label in labels:
    resume_id = label['resume_id']
    job_id = label['job_id']
    
    resume_text = next(r['raw_text'] for r in resumes if r['resume_id'] == resume_id)
    job_text = next(j['raw_text'] for j in jobs if j['job_id'] == job_id)
    
    # Combine resume + job description as input
    combined_text = f"RESUME:\n{resume_text}\n\nJOB:\n{job_text}"
    texts.append(combined_text)
    
    y_labels.append(1 if label['is_match'] else 0)

print(f"Created {len(texts)} training examples")
print(f"Class distribution: {labels_df['is_match'].value_counts().to_dict()}")

# Preprocess texts
print("\nPreprocessing texts...")
preprocessor = ResumePipelinePreprocessor(
    lowercase=True,
    remove_stopwords=True,
    lemmatize=True,
    preserve_skills={
        'python', 'java', 'machine learning', 'docker', 'kubernetes',
        'aws', 'azure', 'sql', 'javascript', 'c++', 'c#'
    }
)

processed_texts = []
for i, text in enumerate(texts):
    processed = preprocessor.preprocess_and_rejoin(text)
    processed_texts.append(processed)
    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(texts)}")

print("✓ Preprocessing complete")

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report, precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# Custom modules
import sys
sys.path.append('../project')
from preprocessing import ResumePipelinePreprocessor

print("✓ All imports successful")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


# Resume Screening System: Model Training & Evaluation

**Goal**: Train a production-grade classifier that:
- Predicts whether a resume matches a job description
- Provides interpretable matches (top keywords)
- Runs fast on CPU (<500ms per batch)
- Generalizes well to unseen resumes

This notebook covers:
1. Data loading & preprocessing
2. Feature engineering (TF-IDF)
3. Model comparison (Logistic Regression vs SVM vs Random Forest)
4. Hyperparameter tuning with GridSearchCV
5. Cross-validation evaluation
6. Final model selection & training
7. Performance metrics & analysis